<a href="https://colab.research.google.com/github/arushk14/arushk14/blob/main/Skin_Disease_Indentifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This project implements a multimodal skin disease diagnosis system using contrastive learning to align image and text embeddings in a shared vector space. Here's a breakdown of the key components:

# Architecture:

1. Dual Encoders:

*   Image Encoder: A ResNet-50 backbone modified to output 512-dimensional embeddings
*   Text Encoder: BERT-based model that encodes clinical descriptions


2. Contrastive Learning Framework:

*   Aligns image and text modalities in a shared embedding space
*   Uses InfoNCE/NT-Xent loss to pull matching image-text pairs closer while pushing non-matching pairs apart


3. Classification Head:

*   Makes final disease predictions based on combined embeddings


# Key Features:

* Multimodal Integration: Leverages both visual and textual information for more accurate diagnoses
* Embedding Visualization: t-SNE visualizations to understand the learned embedding space
* Comprehensive Metrics: Tracks accuracy, precision, recall, and F1 score
* Inference Pipeline: Ready-to-use function for diagnosing new cases

# Data Handling
The code expects a dataset with:

* Images of skin conditions
* Clinical text descriptions
* Disease labels

The SkinDiseaseDataset class handles preprocessing for both modalities, including image transformations and text tokenization.


# Training Process
The model uses a combined loss function:

*   Contrastive loss to align modalities
*   Classification loss for disease prediction






In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from transformers import BertModel, BertTokenizer
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import matplotlib.pyplot as plt
from tqdm import tqdm

In [ ]:
#sees if compatible with CUDA GPU if not then CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

#resize, set to PyTorch Tensors, standardizes each color channel using ImageNet's mean and standard deviation values
image_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) #helps with pretrained weights
])

# Custom dataset
class SkinDiseaseDataset(Dataset):
    def __init__(self, image_dir, annotations_file, transform=None, max_length=128):
        self.image_dir = image_dir
        self.annotations = pd.read_csv(annotations_file)
        self.transform = transform
        self.tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
        self.max_length = max_length
        self.disease_labels = self.annotations['disease_label'].unique()
        self.label_to_id = {label: idx for idx, label in enumerate(self.disease_labels)}

#stores the directory path where images are located
#loads the annotation CSV file into a pandas DataFrame
#stores the image transformation pipeline
#initializes the BERT tokenizer for processing text descriptions
#sets the maximum sequence length for text (128 tokens)
#extracts all unique disease labels from the annotations
#creates a mapping from disease names (strings) to numeric IDs (integers) for classification

    def __len__(self):
        return len(self.annotations) #number of samples in the dataset

    def __getitem__(self, idx):
        img_name = os.path.join(self.image_dir, self.annotations.iloc[idx]['image_filename'])
        image = Image.open(img_name).convert('RGB')

        #image processing
        if self.transform:
            image = self.transform(image)

        #text processing
        description = self.annotations.iloc[idx]['clinical_description'] #retrieves the clinical description text from annotations
        encoded_text = self.tokenizer.encode_plus( #uses the BERT tokenizer to:
            description,
            add_special_tokens=True, #add special tokens ([CLS] and [SEP])
            max_length=self.max_length, #pad or truncate to exactly max_length tokens
            padding='max_length',
            truncation=True,
            return_attention_mask=True, #generate attention masks (1 for real tokens, 0 for padding)
            return_tensors='pt' #convert to PyTorch tensors
        )

        disease_label = self.annotations.iloc[idx]['disease_label']
        label_id = self.label_to_id[disease_label]
        #gets the disease label string from annotations
        #converts it to an integer ID using the mapping created during initialization
        #converts the ID to a PyTorch tensor with the proper data type (long)

        return {
            'image': image,
            'input_ids': encoded_text['input_ids'].squeeze(),
            'attention_mask': encoded_text['attention_mask'].squeeze(),
            'label': torch.tensor(label_id, dtype=torch.long),
            'description': description,
            'disease_name': disease_label
        }
        #returns a dictionary with all the processed data
        #includes both processed tensors needed for model training (image, input_ids, attention_mask, label)
        #also includes the original text and disease name for interpretation and debugging

Using device: cpu


In [ ]:
# Image encoder (CNN)
class ImageEncoder(nn.Module):
    def __init__(self, embedding_dim=512):
        super(ImageEncoder, self).__init__()
        resnet = models.resnet50(pretrained=True)
        modules = list(resnet.children())[:-1]
        self.resnet = nn.Sequential(*modules)
        self.fc = nn.Linear(resnet.fc.in_features, embedding_dim)

# pretrained ResNet-50:

# uses a ResNet-50 model pretrained on ImageNet
# pretrained=True loads weights trained on a large image classification dataset
# this transfer learning approach helps the model learn robust visual features

# list(resnet.children())[:-1] removes the last layer (fully connected classification layer)
# keeps all convolutional and pooling layers that extract image features
# nn.Sequential(*modules) reconstructs the network without the final classification layer

# self.fc is a linear layer that projects the ResNet features to a fixed embedding dimension (default 512)
# allows consistent embedding size across image and text modalities

    def forward(self, x):
        x = self.resnet(x)
        x = x.flatten(start_dim=1)
        x = self.fc(x)
        return F.normalize(x, p=2, dim=1)  # L2 normalize embeddings

# self.resnet(x) passes the input image through the convolutional layers
# generates a feature tensor with shape [batch_size, channels, height, width]

# x.flatten(start_dim=1) converts the feature tensor to a 2D matrix
# combines all spatial and channel dimensions except the batch dimension
# prepares features for the fully connected layer

# self.fc(x) projects features to the target embedding dimension
# learns a linear transformation to create a compact representation

# F.normalize(x, p=2, dim=1) ensures all embeddings have unit length
# helps create a more meaningful embedding space
# ensures consistent scale across different images

# Text encoder (BERT)
class TextEncoder(nn.Module):
    def __init__(self, embedding_dim=512):
        super(TextEncoder, self).__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.fc = nn.Linear(self.bert.config.hidden_size, embedding_dim) # self.fc projects BERT's high-dimensional output to the same embedding space as images

    #CLS is a classification token that summarizes the input sequence and is used to perform classification tasks
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]  # extracts [CLS] token embedding
        x = self.fc(cls_output)
        return F.normalize(x, p=2, dim=1)  # L2 normalize embeddings

In [ ]:
# Classifier head
class ClassificationHead(nn.Module):
    def __init__(self, embedding_dim, num_classes):
        super(ClassificationHead, self).__init__()
        self.fc = nn.Linear(embedding_dim, num_classes) # linear transformation with input of embedding and output of num_classes logits

    def forward(self, x):
        return self.fc(x) #applies linear trans. no activation function applied yet

# Combined multimodal model
# initializes CNN, BERT, ClassificationHead
class MultimodalModel(nn.Module):
    def __init__(self, embedding_dim, num_classes):
        super(MultimodalModel, self).__init__()
        self.image_encoder = ImageEncoder(embedding_dim)
        self.text_encoder = TextEncoder(embedding_dim)
        self.classifier = ClassificationHead(embedding_dim, num_classes)
        self.embedding_dim = embedding_dim

    def forward(self, image, input_ids, attention_mask, get_embeddings=False):
        image_embedding = self.image_encoder(image) # passes image through ResNet-based encoder
        text_embedding = self.text_encoder(input_ids, attention_mask) # passes tokenized text through BERT encoder

        # Combine embeddings (average pooling)
        combined_embedding = (image_embedding + text_embedding) / 2

        if get_embeddings:
            return {
                'image_embedding': image_embedding,
                'text_embedding': text_embedding,
                'combined_embedding': combined_embedding
            }

        # Classification based on combined embedding
        logits = self.classifier(combined_embedding)
        return logits, image_embedding, text_embedding, combined_embedding

In [ ]:
# HELPS MANAGE LOSS

# Contrastive loss (InfoNCE/NT-Xent loss)
class ContrastiveLoss(nn.Module):
    def __init__(self, temperature=0.5):
        super(ContrastiveLoss, self).__init__()
        self.temperature = temperature

    def forward(self, image_embeddings, text_embeddings):
        # Cosine similarity matrix
        logits = torch.matmul(image_embeddings, text_embeddings.T) / self.temperature
        # torch.matmul() computes similarity between all image and text embeddings
        # creates a matrix where each cell represents similarity between an image and text embedding

        # Labels: positive pairs are along the diagonal
        labels = torch.arange(logits.shape[0], device=logits.device)

        # Contrastive loss (symmetric)
        loss_i2t = F.cross_entropy(logits, labels)  # Image to text
        loss_t2i = F.cross_entropy(logits.T, labels)  # Text to image

        return (loss_i2t + loss_t2i) / 2

In [ ]:
# Training function
def train(model, train_loader, optimizer, criterion_cls, criterion_contrastive, epoch):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch in tqdm(train_loader, desc=f"Training Epoch {epoch}"):
        images = batch['image'].to(device)
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()

        # Forward pass
        logits, img_emb, txt_emb, _ = model(images, input_ids, attention_mask)

        # Classification loss
        cls_loss = criterion_cls(logits, labels)

        # Contrastive loss
        cont_loss = criterion_contrastive(img_emb, txt_emb)

        # Combined loss (weighted sum)
        loss = cls_loss + cont_loss

        # Backward pass
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        # Calculate accuracy
        _, predicted = torch.max(logits.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * correct / total

    print(f"Epoch {epoch}: Loss = {epoch_loss:.4f}, Accuracy = {epoch_acc:.2f}%", flush=True)
    return epoch_loss, epoch_acc

In [ ]:
# Evaluation function
# tqdm helps with loops
def evaluate(model, val_loader, criterion_cls, criterion_contrastive):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    with torch.no_grad(): #batch processing loop
        for batch in tqdm(val_loader, desc="Evaluating"):
            images = batch['image'].to(device)
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            # Forward pass
            logits, img_emb, txt_emb, _ = model(images, input_ids, attention_mask)

            # Classification loss
            cls_loss = criterion_cls(logits, labels)

            # Contrastive loss
            cont_loss = criterion_contrastive(img_emb, txt_emb)

            # Combined loss
            loss = cls_loss + cont_loss

            running_loss += loss.item()

            # Calculate accuracy
            _, predicted = torch.max(logits.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    val_loss = running_loss / len(val_loader)
    val_acc = 100 * correct / total

    # Calculate precision, recall, and F1
    precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted')

    print(f"Validation: Loss = {val_loss:.4f}, Accuracy = {val_acc:.2f}%, Precision = {precision:.4f}, Recall = {recall:.4f}, F1 = {f1:.4f}")
    return val_loss, val_acc, precision, recall, f1

In [ ]:
# Visualize embeddings using t-SNE
def visualize_embeddings(model, data_loader, num_samples=100):
    from sklearn.manifold import TSNE

    model.eval()
    image_embeddings = []
    text_embeddings = []
    combined_embeddings = []
    labels = []
    disease_names = []


    # Extract embeddings from a trained model
    # Collect a subset of embeddings for further analysis
    # Prepare data for dimensionality reduction (t-SNE)

    with torch.no_grad():
        for batch in data_loader:
            if len(labels) >= num_samples:
                break

            images = batch['image'].to(device)
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            batch_labels = batch['label']
            batch_diseases = batch['disease_name']

            embeddings = model(images, input_ids, attention_mask, get_embeddings=True)

            image_embeddings.append(embeddings['image_embedding'].cpu().numpy())
            text_embeddings.append(embeddings['text_embedding'].cpu().numpy())
            combined_embeddings.append(embeddings['combined_embedding'].cpu().numpy())
            labels.append(batch_labels.numpy())
            disease_names.extend(batch_diseases)

            #Embedding Conversion:
            # .cpu(): Moves tensors to CPU
            # .numpy(): Converts to NumPy arrays
            # Allows compatibility with visualization libraries

            # Collection Methods:
            # append(): Adds embeddings to lists
            # extend(): Adds disease names


            if len(labels) >= num_samples:
                break

    # Concatenate embeddings and labels
    image_embeddings = np.vstack(image_embeddings)[:num_samples]
    text_embeddings = np.vstack(text_embeddings)[:num_samples]
    combined_embeddings = np.vstack(combined_embeddings)[:num_samples]
    labels = np.concatenate(labels)[:num_samples]
    disease_names = disease_names[:num_samples]

    # Apply t-SNE
    tsne = TSNE(n_components=2, random_state=42)

    img_tsne = tsne.fit_transform(image_embeddings)
    txt_tsne = tsne.fit_transform(text_embeddings)
    combined_tsne = tsne.fit_transform(combined_embeddings)

    # Plot
    plt.figure(figsize=(18, 6))

    # Image embeddings
    plt.subplot(1, 3, 1)
    scatter = plt.scatter(img_tsne[:, 0], img_tsne[:, 1], c=labels, cmap='tab10', alpha=0.7)
    plt.title('Image Embeddings')
    plt.colorbar(scatter)

    # Text embeddings
    plt.subplot(1, 3, 2)
    scatter = plt.scatter(txt_tsne[:, 0], txt_tsne[:, 1], c=labels, cmap='tab10', alpha=0.7)
    plt.title('Text Embeddings')
    plt.colorbar(scatter)

    # Combined embeddings
    plt.subplot(1, 3, 3)
    scatter = plt.scatter(combined_tsne[:, 0], combined_tsne[:, 1], c=labels, cmap='tab10', alpha=0.7)
    plt.title('Combined Embeddings')
    plt.colorbar(scatter)

    plt.tight_layout()
    plt.savefig('embeddings_visualization.png')
    plt.close()

    # Create a separate plot with disease names
    plt.figure(figsize=(10, 8))
    unique_labels = np.unique(labels)
    colors = plt.cm.tab10(np.linspace(0, 1, len(unique_labels)))

    for i, label in enumerate(unique_labels):
        idx = labels == label
        plt.scatter(combined_tsne[idx, 0], combined_tsne[idx, 1], color=colors[i], label=disease_names[np.where(idx)[0][0]], alpha=0.7)

    plt.legend()
    plt.title('Combined Embeddings by Disease Type')
    plt.savefig('disease_embeddings.png')
    plt.show()

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

def prepare_dataset(base_folder, max_samples_per_class=500):
    """
    Prepares dataset from a folder structure with disease subfolders
    Args:
    base_folder (str): Path to main folder containing disease subfolders
    max_samples_per_class (int): Maximum number of samples per disease class
    Returns:
    pd.DataFrame: Annotations dataframe
    """
    annotations = []
    # Print to verify the base folder exists
    print(f"Looking for disease folders in: {base_folder}")
    if not os.path.exists(base_folder):
        print(f"WARNING: Base folder {base_folder} does not exist!")
        return pd.DataFrame()

    # Iterate through disease subfolders
    for disease_folder in os.listdir(base_folder):
        disease_path = os.path.join(base_folder, disease_folder)
        # Skip if not a directory
        if not os.path.isdir(disease_path):
            continue
        # Collect image files
        image_files = [f for f in os.listdir(disease_path)
                       if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff'))]
        # Limit samples per class
        image_files = image_files[:max_samples_per_class]
        # Create basic clinical descriptions
        for image_file in image_files:
            annotations.append({
                'image_filename': os.path.join(disease_folder, image_file),
                'clinical_description': f'Clinical image of {disease_folder} skin condition',
                'disease_label': disease_folder
            })
    # Convert to DataFrame
    annotations_df = pd.DataFrame(annotations)

    if annotations_df.empty:
        print("WARNING: No valid image files found in the disease subfolders!")
    else:
        # Print the columns of the DataFrame
        print("Columns in annotations_df:", annotations_df.columns)
        # Print some sample data to check the structure
        print("Sample data:", annotations_df.head())
        print(f"Total images found: {len(annotations_df)}")

    return annotations_df

def split_dataset(annotations_df, train_ratio=0.8):
    """
    Splits dataset into training and validation sets
    Args:
    annotations_df (pd.DataFrame): Full annotations dataframe
    train_ratio (float): Proportion of data for training
    Returns:
    tuple: (train_df, val_df)
    """
    # Group by disease label to ensure balanced split
    train_data = []
    val_data = []
    for disease_label in annotations_df['disease_label'].unique():
        disease_subset = annotations_df[annotations_df['disease_label'] == disease_label]
        # Compute split index
        split_idx = int(len(disease_subset) * train_ratio)
        # Split data
        disease_train = disease_subset.iloc[:split_idx]
        disease_val = disease_subset.iloc[split_idx:]
        train_data.append(disease_train)
        val_data.append(disease_val)
    # Combine and shuffle
    train_df = pd.concat(train_data).sample(frac=1)
    val_df = pd.concat(val_data).sample(frac=1)
    return train_df, val_df

# Define missing classes and functions (placeholders for your actual implementations)
class SkinDiseaseDataset:
    def __init__(self, image_dir, annotations_file, transform=None):
        self.image_dir = image_dir
        self.transform = transform
        self.annotations = pd.read_csv(annotations_file)
        self.disease_labels = self.annotations['disease_label'].unique()
        self.label_to_id = {label: idx for idx, label in enumerate(self.disease_labels)}
        self.tokenizer = None  # You would initialize your actual tokenizer here

class MultimodalModel(nn.Module):
    def __init__(self, embedding_dim, num_classes):
        super(MultimodalModel, self).__init__()
        # Your model implementation here
        self.embedding_dim = embedding_dim
        self.num_classes = num_classes

class ContrastiveLoss(nn.Module):
    def __init__(self, temperature=0.5):
        super(ContrastiveLoss, self).__init__()
        self.temperature = temperature

    def forward(self, x1, x2):
        # Your contrastive loss implementation here
        return torch.tensor(0.0)

def train(model, data_loader, optimizer, criterion_cls, criterion_contrastive, epoch):
    # Your training logic here
    return 0.5, 80.0  # Dummy return values for loss and accuracy

def evaluate(model, data_loader, criterion_cls, criterion_contrastive):
    # Your evaluation logic here
    return 0.6, 75.0, 0.74, 0.73, 0.72  # Dummy return values

def visualize_embeddings(model, data_loader):
    # Your embedding visualization logic here
    pass

def inference(model, img_path, description, transforms, tokenizer, device, label_to_id):
    # Your inference logic here
    return {
        'primary_diagnosis': 'Example',
        'confidence': 0.8,
        'top_predictions': [('Example1', 0.8), ('Example2', 0.1), ('Example3', 0.05)]
    }

# Define image transforms (placeholder)
image_transforms = None

# Define device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Print current working directory for debugging
print("Current working directory:", os.getcwd())

# Define base folder consistently with full path
base_folder = '/content/drive/MyDrive/data'

# Create data directories
os.makedirs('data', exist_ok=True)
os.makedirs('data/skin_disease_images', exist_ok=True)

# Process dataset
print("Processing dataset...")
annotations_df = prepare_dataset(base_folder)

# Check if annotations_df is empty before splitting
if annotations_df.empty:
    print("Error: annotations_df is empty. Check your dataset and file paths.")
else:
    # Split dataset
    train_df, val_df = split_dataset(annotations_df)

    # Save to CSV for use in main model
    train_csv_path = 'data/train_annotations.csv'
    val_csv_path = 'data/val_annotations.csv'

    train_df.to_csv(train_csv_path, index=False)
    val_df.to_csv(val_csv_path, index=False)

    print(f"Saved training annotations to: {train_csv_path}")
    print(f"Saved validation annotations to: {val_csv_path}")
    print(f"Training samples: {len(train_df)}")
    print(f"Validation samples: {len(val_df)}")
    print("Disease classes:", train_df['disease_label'].unique())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cpu
Current working directory: /content
Processing dataset...
Looking for disease folders in: /content/drive/MyDrive/data
Error: annotations_df is empty. Check your dataset and file paths.


In [ ]:
# Main training and evaluation function
def main():
    # Check if annotation files exist before proceeding
    if not os.path.exists('data/train_annotations.csv') or not os.path.exists('data/val_annotations.csv'):
        print("Error: Annotation files not found. Please run the dataset preparation steps first.")
        return

    # Hyperparameters
    batch_size = 16
    learning_rate = 2e-5
    num_epochs = 20
    embedding_dim = 512

    # Path to data directory
    data_dir = "data"  # Main folder containing disease subfolders

    # Create datasets from folder structure
    train_dataset, val_dataset = create_dataset_from_folders(data_dir)

    # Get number of classes
    num_classes = len(train_dataset.disease_labels)
    print(f"Number of disease classes: {num_classes}")

    # Create data loaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

    # Initialize model
    model = MultimodalModel(embedding_dim, num_classes).to(device)

    # Define loss functions
    criterion_cls = nn.CrossEntropyLoss()
    criterion_contrastive = ContrastiveLoss(temperature=0.5)

    # Define optimizer
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

    # Learning rate scheduler
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

    # Training loop
    train_losses = []
    val_losses = []
    train_accs = []
    val_accs = []
    best_val_f1 = 0.0

    for epoch in range(1, num_epochs + 1):
        # Train
        train_loss, train_acc = train(model, train_loader, optimizer, criterion_cls, criterion_contrastive, epoch)
        train_losses.append(train_loss)
        train_accs.append(train_acc)

        # Evaluate
        val_loss, val_acc, precision, recall, f1 = evaluate(model, val_loader, criterion_cls, criterion_contrastive)
        val_losses.append(val_loss)
        val_accs.append(val_acc)

        # Update learning rate
        scheduler.step(val_loss)

        # Save best model
        if f1 > best_val_f1:
            best_val_f1 = f1
            torch.save(model.state_dict(), 'best_skin_disease_model.pth')
            print(f"Saved new best model with F1 score: {f1:.4f}")

    # Load best model
    model.load_state_dict(torch.load('best_skin_disease_model.pth'))

    # Visualize embeddings
    visualize_embeddings(model, val_loader)

    # Plot training curves
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(range(1, num_epochs + 1), train_losses, label='Train Loss')
    plt.plot(range(1, num_epochs + 1), val_losses, label='Val Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(range(1, num_epochs + 1), train_accs, label='Train Acc')
    plt.plot(range(1, num_epochs + 1), val_accs, label='Val Acc')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy (%)')
    plt.legend()

    plt.tight_layout()
    plt.savefig('training_curves.png')
    plt.show()

    # Final evaluation
    print("\nFinal Evaluation on Validation Set:")
    _, _, precision, recall, f1 = evaluate(model, val_loader, criterion_cls, criterion_contrastive)

    print("\nExample inference:")
    example_img_path = "data/skin_disease_images/example.jpg"
    example_description = "Reddish patch with distinct border and silver scales on the elbow."

    if os.path.exists(example_img_path):
        result = inference(
            model,
            example_img_path,
            example_description,
            image_transforms,
            train_dataset.tokenizer,
            device,
            train_dataset.label_to_id
        )
        print(f"Primary diagnosis: {result['primary_diagnosis']} (Confidence: {result['confidence']:.4f})")
        print("Top 3 predictions:")
        for disease, prob in result['top_predictions']:
            print(f"- {disease}: {prob:.4f}")
    else:
        print(f"Example image not found at {example_img_path}")

if __name__ == "__main__":
    main()

Error: Annotation files not found. Please run the dataset preparation steps first.


In [ ]:
# Inference function for new samples
# Takes an image and its clinical description
# Generates predictions for skin disease diagnosis
# Extracts and returns model embeddings
# Provides top-k predictions with probabilities

def inference(model, image_path, description, transform, tokenizer, device, label_map):
    model.eval()

    # Process image
    image = Image.open(image_path).convert('RGB')
    image = transform(image).unsqueeze(0).to(device)

    # Process text
    encoded_text = tokenizer.encode_plus(
        description,
        add_special_tokens=True,
        max_length=128,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt'
    )
    input_ids = encoded_text['input_ids'].to(device)
    attention_mask = encoded_text['attention_mask'].to(device)

    with torch.no_grad():
        # Get predictions and embeddings
        logits, img_emb, txt_emb, combined_emb = model(image, input_ids, attention_mask)
        probs = F.softmax(logits, dim=1)
        confidence, predicted = torch.max(probs, 1)

        # Get top 3 predictions
        values, indices = torch.topk(probs, 3)

    # Convert to label names
    id_to_label = {v: k for k, v in label_map.items()}
    top_predictions = [(id_to_label[idx.item()], val.item()) for idx, val in zip(indices[0], values[0])]

    return {
        'top_predictions': top_predictions,
        'primary_diagnosis': id_to_label[predicted.item()],
        'confidence': confidence.item(),
        'image_embedding': img_emb.cpu().numpy(),
        'text_embedding': txt_emb.cpu().numpy(),
        'combined_embedding': combined_emb.cpu().numpy()
    }